# GCRA: Grid-Centric Rotational Analysis — Fingerprint Recognition

**Author:** Akhilesh Verma | Department of CSE, AKGEC Ghaziabad | vermaakhilesh@akgec.ac.in

**Paper:** *A Statistical Grid and Rotational Analysis Model for Fingerprint Recognition*

---

## Notebook Structure

| Part | Description |
|------|-------------|
| **1** | Synthetic Fingerprint Data Generator |
| **2** | Full GCRA Pipeline (Grid + Rotation + Statistics) |
| **3** | Matching and Similarity Scoring |
| **4** | Visualisations (6 publication-quality figures) |

### Requirements
```
pip install numpy matplotlib scipy pandas seaborn
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
import pandas as pd
from scipy.stats import skew, kurtosis
from itertools import product

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
print('All imports successful. Seed:', RANDOM_SEED)

---
## Part 1 — Synthetic Fingerprint Data Generator

Generates synthetic minutiae sets mimicking real fingerprint statistics.

> **To use real data:** replace `generate_scan()` with a loader that reads `.xyt` FVC files and returns the same `(N, 4)` array format: `[x, y, angle, type]`

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────
IMAGE_W     = 200
IMAGE_H     = 200
N_MINUTIAE  = 25
N_SCANS     = 5
NOISE_STD   = 5.0
CLUSTER_STD = 30.0


def generate_finger_template(n_minutiae=N_MINUTIAE, image_w=IMAGE_W,
                              image_h=IMAGE_H, cluster_std=CLUSTER_STD, seed=None):
    """
    Generate ground-truth minutiae for one finger.
    Returns ndarray (n_minutiae, 4): [x, y, angle_deg, type]
    type: 0=ridge_ending, 1=bifurcation
    """
    rng = np.random.RandomState(seed)
    cx, cy = image_w / 2, image_h / 2
    x = np.clip(rng.normal(cx, cluster_std, n_minutiae), 5, image_w - 5)
    y = np.clip(rng.normal(cy, cluster_std, n_minutiae), 5, image_h - 5)
    angles = rng.uniform(0, 180, n_minutiae)
    types  = rng.randint(0, 2, n_minutiae)
    return np.column_stack([x, y, angles, types])


def generate_scan(template, noise_std=NOISE_STD, seed=None):
    """
    Simulate one scan by adding Gaussian placement noise to template.
    Mimics inter-scan variability from pressure and positioning.
    """
    rng  = np.random.RandomState(seed)
    scan = template.copy()
    scan[:, 0] += rng.normal(0, noise_std, len(template))
    scan[:, 1] += rng.normal(0, noise_std, len(template))
    scan[:, 0]  = np.clip(scan[:, 0], 0, IMAGE_W - 1)
    scan[:, 1]  = np.clip(scan[:, 1], 0, IMAGE_H - 1)
    return scan


def generate_dataset(n_fingers=5, n_scans=N_SCANS, seed=RANDOM_SEED):
    """
    Generate complete synthetic dataset.
    Returns dict {finger_id: [scan_0, ..., scan_{n_scans-1}]}
    """
    dataset = {}
    for fid in range(n_fingers):
        template = generate_finger_template(seed=seed + fid)
        dataset[fid] = [generate_scan(template, seed=seed + fid*100 + s)
                        for s in range(n_scans)]
    return dataset


# ── Generate dataset ──────────────────────────────────────────────────
dataset = generate_dataset(n_fingers=5, n_scans=N_SCANS)

print(f'Dataset: {len(dataset)} fingers, {N_SCANS} scans each')
print(f'Scan shape: {dataset[0][0].shape} (x, y, angle, type)')
print('\nSample — Finger-0, Scan-0 (first 5 rows):')
df = pd.DataFrame(dataset[0][0], columns=['x','y','angle','type'])
df['type'] = df['type'].map({0:'ending', 1:'bifurcation'})
print(df.head().round(2).to_string(index=False))

---
## Part 2 — Full GCRA Pipeline

Implements all five stages from the paper:

1. Bounding-box normalisation (Eq. 1)
2. Centroid computation (Eq. 2)
3. Grid overlay with rotational accumulation (Eq. 3–5)
4. Statistical descriptors: variance, skewness, kurtosis (Eq. 6–8)
5. Feature vector **f** ∈ ℝ⁹ (Eq. 9)

In [ ]:
# ── Pipeline configuration ─────────────────────────────────────────────
GRID_SIZES    = [3, 5, 7]
ROTATION_STEP = 5
ROTATIONS     = np.arange(0, 360, ROTATION_STEP)  # 72 steps


def bounding_box_normalise(minutiae, target_w=IMAGE_W, target_h=IMAGE_H):
    """Normalise minutiae to [0,W]x[0,H] using bounding box. (Eq. 1)"""
    x, y   = minutiae[:, 0], minutiae[:, 1]
    xmin, xmax = x.min(), x.max()
    ymin, ymax = y.min(), y.max()
    dx = xmax - xmin if xmax != xmin else 1.0
    dy = ymax - ymin if ymax != ymin else 1.0
    norm = minutiae.copy().astype(float)
    norm[:, 0] = (x - xmin) / dx * target_w
    norm[:, 1] = (y - ymin) / dy * target_h
    return norm, dict(xmin=xmin, xmax=xmax, ymin=ymin, ymax=ymax)


def compute_centroid(minutiae):
    """Geometric centroid of minutiae. (Eq. 2: Cx=mean(xi), Cy=mean(yi))"""
    return float(np.mean(minutiae[:, 0])), float(np.mean(minutiae[:, 1]))


def rotation_matrix_2d(phi_deg):
    """2-D rotation matrix for angle phi (degrees). (Eq. 4)"""
    phi = np.radians(phi_deg)
    return np.array([[np.cos(phi), -np.sin(phi)],
                     [np.sin(phi),  np.cos(phi)]])


def rotational_accumulation(minutiae, K,
                             rotations=ROTATIONS,
                             image_w=IMAGE_W, image_h=IMAGE_H):
    """
    Core GCRA step: overlay K×K grid, rotate 0→355° in 5° steps,
    accumulate per-cell minutiae counts. (Eq. 3, 5)

    Returns
    -------
    A          : ndarray (K, K) — accumulated count matrix A^(K)
    all_counts : list of (K,K) arrays — per-rotation counts
    cx, cy     : centroid coordinates
    """
    norm, _    = bounding_box_normalise(minutiae, image_w, image_h)
    cx, cy     = compute_centroid(norm)
    half_w     = image_w / 2
    half_h     = image_h / 2
    xy_centred = norm[:, :2] - np.array([cx, cy])   # (n, 2)

    A          = np.zeros((K, K), dtype=int)
    all_counts = []

    for phi in rotations:
        R      = rotation_matrix_2d(phi)
        xy_rot = (R @ xy_centred.T).T            # (n, 2)
        G      = np.zeros((K, K), dtype=int)

        for xr, yr in xy_rot:
            col_f = (xr + half_w) / (2 * half_w) * K
            row_f = (yr + half_h) / (2 * half_h) * K
            col   = int(np.floor(col_f))
            row   = int(np.floor(row_f))
            if 0 <= row < K and 0 <= col < K:
                G[row, col] += 1

        A += G
        all_counts.append(G.copy())

    return A, all_counts, cx, cy


def compute_descriptors(A):
    """
    Compute variance (Eq. 6), skewness (Eq. 7), excess kurtosis (Eq. 8)
    from accumulated count matrix A.
    """
    a = A.flatten().astype(float)
    if a.std() == 0:
        return 0.0, 0.0, 0.0
    return (float(np.var(a)),
            float(skew(a, bias=False)),
            float(kurtosis(a, fisher=True, bias=False)))


def gcra_pipeline(minutiae, grid_sizes=GRID_SIZES):
    """
    Full GCRA pipeline for one scan.

    Returns
    -------
    f       : ndarray (9,) — feature vector [d^(3), d^(5), d^(7)] (Eq. 9)
    details : dict — intermediate results keyed by K
    """
    parts, details = [], {}
    for K in grid_sizes:
        A, counts, cx, cy = rotational_accumulation(minutiae, K)
        var, sk, kurt     = compute_descriptors(A)
        parts.extend([var, sk, kurt])
        details[K] = dict(A=A, all_counts=counts, cx=cx, cy=cy,
                          variance=var, skewness=sk, kurtosis=kurt)
    return np.array(parts), details


# ── Run on all scans ──────────────────────────────────────────────────
all_features = {fid: [] for fid in dataset}
all_details  = {fid: [] for fid in dataset}

for fid, scans in dataset.items():
    for scan in scans:
        fvec, det = gcra_pipeline(scan)
        all_features[fid].append(fvec)
all_details = {fid: [gcra_pipeline(scan)[1] for scan in scans]
               for fid, scans in dataset.items()}

labels_9 = ['3x3_var','3x3_skew','3x3_kurt',
             '5x5_var','5x5_skew','5x5_kurt',
             '7x7_var','7x7_skew','7x7_kurt']

print('GCRA pipeline complete.')
print(f'Feature vector: {len(all_features[0][0])} dimensions (3 grids x 3 stats)')
print('\nFeature vectors for Finger-0:')
df_fv = pd.DataFrame(all_features[0], columns=labels_9)
df_fv.index.name = 'scan'
print(df_fv.round(3).to_string())

In [ ]:
# ── Statistical descriptors table (replicates Table 10 from paper) ──
rows = []
for fid in range(len(dataset)):
    for sid in range(N_SCANS):
        row = {'Finger': fid, 'Scan': sid}
        for K in GRID_SIZES:
            d = all_details[fid][sid][K]
            row[f'{K}x{K}_Var']  = round(d['variance'],  2)
            row[f'{K}x{K}_Skew'] = round(d['skewness'],  3)
            row[f'{K}x{K}_Kurt'] = round(d['kurtosis'],  3)
        rows.append(row)

df_stats = pd.DataFrame(rows)
print('Statistical Descriptors — All Fingers, All Scans')
print('='*90)
print(df_stats.to_string(index=False))

---
## Part 3 — Matching and Similarity Scoring

- **Euclidean similarity**: `s(f_q, f_s) = −‖f_q − f_s‖₂` (Eq. 9 of paper)
- **Cosine similarity**: additional metric
- Full genuine vs. impostor score distribution computed

In [ ]:
def euclidean_sim(f1, f2):
    """Negative Euclidean distance. Higher = more similar. (Eq. 9)"""
    return -float(np.linalg.norm(f1 - f2))


def cosine_sim(f1, f2):
    """Cosine similarity in [-1, 1]. Higher = more similar."""
    d = np.linalg.norm(f1) * np.linalg.norm(f2)
    return float(np.dot(f1, f2) / d) if d > 0 else 0.0


def build_score_matrix(all_features, metric='euclidean'):
    """
    Compute all pairwise similarity scores.
    Returns genuine_scores, impostor_scores, records (list of dicts)
    """
    fn = euclidean_sim if metric == 'euclidean' else cosine_sim
    entries = [(fid, sid, fv)
               for fid, fvecs in all_features.items()
               for sid, fv in enumerate(fvecs)]
    genuine, impostor, records = [], [], []
    for i in range(len(entries)):
        for j in range(i+1, len(entries)):
            fi, si, fvi = entries[i]
            fj, sj, fvj = entries[j]
            score = fn(fvi, fvj)
            label = 'genuine' if fi == fj else 'impostor'
            (genuine if fi == fj else impostor).append(score)
            records.append({'Finger_A':fi,'Scan_A':si,
                             'Finger_B':fj,'Scan_B':sj,
                             'Score':round(score,4),'Label':label})
    return genuine, impostor, records


def identify(query_fv, enrolled, metric='euclidean'):
    """
    Identify best-matching finger for a query feature vector.
    enrolled = {finger_id: mean_feature_vector}
    """
    fn = euclidean_sim if metric == 'euclidean' else cosine_sim
    scores = {fid: fn(query_fv, tmpl) for fid, tmpl in enrolled.items()}
    return max(scores, key=scores.get), scores


# ── Enrolled templates = mean feature vector per finger ───────────────
enrolled = {fid: np.mean(fvecs, axis=0)
            for fid, fvecs in all_features.items()}

genuine, impostor, records = build_score_matrix(all_features)
df_scores = pd.DataFrame(records)

print(f'Total pairs: {len(records)} | Genuine: {len(genuine)} | Impostor: {len(impostor)}')
print(f'Genuine  — mean: {np.mean(genuine):.3f}, std: {np.std(genuine):.3f}')
print(f'Impostor — mean: {np.mean(impostor):.3f}, std: {np.std(impostor):.3f}')

In [ ]:
print('Identification Results (Rank-1)')
print('='*45)
print(f'{"Finger":>6} {"Scan":>4} {"Predicted":>9} {"OK":>4}')
print('-'*45)
correct, total = 0, 0
for fid, fvecs in all_features.items():
    for sid, fv in enumerate(fvecs):
        pred, _ = identify(fv, enrolled)
        ok = (pred == fid)
        correct += int(ok)
        total   += 1
        print(f'{fid:>6} {sid:>4} {pred:>9} {"✓" if ok else "✗":>4}')
print('-'*45)
print(f'Rank-1 Accuracy (synthetic): {correct}/{total} = {100*correct/total:.1f}%')
print('NOTE: Accuracy on synthetic data ONLY — not indicative of real-world performance.')

---
## Part 4 — Visualisations

Six publication-quality figures:
1. Simulated scans with centroid (≈ Figure 1 of paper)
2. Accumulated grid heatmaps (≈ Figure 2 of paper)
3. Statistical descriptor bar charts
4. Feature vector heatmap (all fingers × all scans)
5. Genuine vs. impostor score distributions
6. Pairwise similarity score matrix

In [ ]:
# ── Figure 1: Simulated Scans ─────────────────────────────────────────
fig, axes = plt.subplots(1, N_SCANS, figsize=(15, 3))
fig.suptitle('Figure 1 — Five Simulated Fingerprint Scans (Finger 0)\n'
             'Blue circles = ridge endings | Blue triangles = bifurcations | Red X = centroid',
             fontsize=11, fontweight='bold', y=1.08)

for sid, ax in enumerate(axes):
    scan = dataset[0][sid]
    norm, _ = bounding_box_normalise(scan)
    cx, cy  = compute_centroid(norm)
    ax.set_xlim(0, IMAGE_W); ax.set_ylim(0, IMAGE_H)
    ax.set_aspect('equal'); ax.set_facecolor('#f5f5f5')
    for v in np.linspace(0, IMAGE_W, 8):
        ax.axvline(v, color='gray', lw=0.3, alpha=0.4)
        ax.axhline(v, color='gray', lw=0.3, alpha=0.4)
    end = norm[norm[:, 3] == 0]
    bif = norm[norm[:, 3] == 1]
    ax.scatter(end[:, 0], end[:, 1], c='steelblue', s=35, zorder=3, label='Ending')
    ax.scatter(bif[:, 0], bif[:, 1], c='navy', s=35, marker='^', zorder=3, label='Bifurcation')
    ax.scatter(cx, cy, c='red', s=140, marker='X', zorder=5, label='Centroid')
    ax.set_title(f'Scan {sid}', fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
axes[0].legend(fontsize=7, loc='upper left')
plt.tight_layout()
plt.savefig('fig1_simulated_scans.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig1_simulated_scans.png')

In [ ]:
# ── Figure 2: Accumulated Heatmaps ───────────────────────────────────
fig, axes = plt.subplots(N_SCANS, len(GRID_SIZES),
                          figsize=(len(GRID_SIZES)*3.5, N_SCANS*3))
fig.suptitle('Figure 2 — Accumulated Minutiae Count Heatmaps A^(K)\n'
             'Summed over 72 rotational steps (0°→355°, step 5°) | Finger 0',
             fontsize=11, fontweight='bold')

for sid in range(N_SCANS):
    for ki, K in enumerate(GRID_SIZES):
        ax = axes[sid][ki]
        A  = all_details[0][sid][K]['A']
        im = ax.imshow(A, cmap='YlOrRd', interpolation='nearest')
        for r, c in product(range(K), range(K)):
            ax.text(c, r, str(A[r,c]), ha='center', va='center',
                    fontsize=8 if K<=5 else 6, fontweight='bold')
        ax.set_title(f'Scan {sid} | {K}x{K}', fontsize=8)
        ax.set_xticks([]); ax.set_yticks([])
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.savefig('fig2_heatmaps.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig2_heatmaps.png')

In [ ]:
# ── Figure 3: Statistical Descriptors Bar Chart ───────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Figure 3 — Statistical Descriptors per Grid Size (Finger 0)',
             fontsize=11, fontweight='bold')

stat_info = [('Variance', 'variance'), ('Skewness', 'skewness'), ('Excess Kurtosis', 'kurtosis')]
colors = ['#2196F3', '#FF5722', '#4CAF50']
x = np.arange(N_SCANS)
w = 0.25

for idx, (title, key) in enumerate(stat_info):
    ax = axes[idx]
    for ki, K in enumerate(GRID_SIZES):
        vals = [all_details[0][sid][K][key] for sid in range(N_SCANS)]
        ax.bar(x + ki*w, vals, width=w, label=f'{K}x{K}',
               color=colors[ki], alpha=0.82, edgecolor='black', lw=0.5)
    ax.set_xlabel('Scan', fontsize=10)
    ax.set_ylabel(title, fontsize=10)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xticks(x + w); ax.set_xticklabels([f'S{i}' for i in range(N_SCANS)])
    ax.legend(title='Grid', fontsize=8)
    ax.axhline(0, color='black', lw=0.7, ls='--'); ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('fig3_stat_descriptors.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig3_stat_descriptors.png')

In [ ]:
# ── Figure 4: Feature Vector Heatmap ─────────────────────────────────
fv_mat  = np.array([all_features[fid][sid]
                    for fid in range(len(dataset))
                    for sid in range(N_SCANS)])
row_lbl = [f'F{fid}-S{sid}'
           for fid in range(len(dataset))
           for sid in range(N_SCANS)]

fv_norm = (fv_mat - fv_mat.min(0)) / (fv_mat.ptp(0) + 1e-9)

fig, ax = plt.subplots(figsize=(12, 6))
im = ax.imshow(fv_norm, cmap='viridis', aspect='auto')
ax.set_xticks(range(9)); ax.set_xticklabels(labels_9, rotation=35, ha='right', fontsize=9)
ax.set_yticks(range(len(row_lbl))); ax.set_yticklabels(row_lbl, fontsize=8)
ax.set_title('Figure 4 — GCRA Feature Vector Heatmap\n'
             '(All fingers x all scans, normalised [0,1] per dimension)',
             fontsize=11, fontweight='bold')
for fid in range(1, len(dataset)):
    ax.axhline(fid*N_SCANS - 0.5, color='white', lw=2)
plt.colorbar(im, ax=ax, label='Normalised value')
plt.tight_layout()
plt.savefig('fig4_feature_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig4_feature_heatmap.png')

In [ ]:
# ── Figure 5: Genuine vs Impostor Score Distribution ──────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Figure 5 — Genuine vs. Impostor Score Distributions\n'
             '(Euclidean distance metric | synthetic data only)',
             fontsize=11, fontweight='bold')

ax1.hist(genuine,  bins=20, alpha=0.72, color='green',  edgecolor='darkgreen', label=f'Genuine (n={len(genuine)})')
ax1.hist(impostor, bins=20, alpha=0.72, color='red',    edgecolor='darkred',   label=f'Impostor (n={len(impostor)})')
ax1.axvline(np.mean(genuine),  color='darkgreen', ls='--', lw=2, label=f'Mean G = {np.mean(genuine):.2f}')
ax1.axvline(np.mean(impostor), color='darkred',   ls='--', lw=2, label=f'Mean I = {np.mean(impostor):.2f}')
ax1.set_xlabel('Similarity Score (−Euclidean distance)', fontsize=10)
ax1.set_ylabel('Frequency', fontsize=10)
ax1.set_title('Score Histogram', fontsize=11)
ax1.legend(fontsize=8); ax1.grid(alpha=0.3)

bp = ax2.boxplot([genuine, impostor], labels=['Genuine','Impostor'],
                  patch_artist=True, notch=False)
bp['boxes'][0].set_facecolor('lightgreen')
bp['boxes'][1].set_facecolor('lightcoral')
ax2.set_ylabel('Similarity Score', fontsize=10)
ax2.set_title('Score Box Plot', fontsize=11)
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('fig5_score_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig5_score_distribution.png')

In [ ]:
# ── Figure 6: Pairwise Similarity Score Matrix ────────────────────────
n_total  = len(dataset) * N_SCANS
flat_fv  = [(fid, sid, all_features[fid][sid])
             for fid in range(len(dataset)) for sid in range(N_SCANS)]
score_mat = np.array([[euclidean_sim(fvi, fvj)
                        for (_, _, fvj) in flat_fv]
                       for (_, _, fvi) in flat_fv])
rlabels   = [f'F{fi}S{si}' for fi, si, _ in flat_fv]

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(score_mat, cmap='RdYlGn', interpolation='nearest')
ax.set_title('Figure 6 — Pairwise Similarity Score Matrix\n'
             'Blue lines = finger boundaries | Diagonal = 0',
             fontsize=11, fontweight='bold')
ax.set_xticks(range(n_total)); ax.set_xticklabels(rlabels, rotation=90, fontsize=7)
ax.set_yticks(range(n_total)); ax.set_yticklabels(rlabels, fontsize=7)
for fid in range(1, len(dataset)):
    ax.axhline(fid*N_SCANS - 0.5, color='blue', lw=1.5)
    ax.axvline(fid*N_SCANS - 0.5, color='blue', lw=1.5)
plt.colorbar(im, ax=ax, label='Similarity Score')
plt.tight_layout()
plt.savefig('fig6_score_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig6_score_matrix.png')

---
## Summary

| Component | Status |
|-----------|--------|
| Synthetic data generator | ✅ |
| Bounding-box normalisation (Eq. 1) | ✅ |
| Centroid computation (Eq. 2) | ✅ |
| Grid overlay + rotational accumulation (Eq. 3–5) | ✅ |
| Variance, Skewness, Kurtosis (Eq. 6–8) | ✅ |
| Feature vector f ∈ ℝ⁹ (Eq. 9) | ✅ |
| Euclidean + cosine matching | ✅ |
| Genuine vs. impostor scoring | ✅ |
| 6 publication-quality figures | ✅ |

---
### To use with real FVC data

```python
def load_fvc_xyt(filepath):
    """
    Load FVC .xyt minutiae file.
    Format: x  y  angle (one minutia per line)
    """
    data  = np.loadtxt(filepath)
    types = np.zeros(len(data))          # FVC files don't include type
    return np.column_stack([data[:, :2], data[:, 2], types])

# Then use exactly as before:
# minutiae = load_fvc_xyt('fvc2000_db1_001_1.xyt')
# fvec, details = gcra_pipeline(minutiae)
```

---
*For questions: vermaakhilesh@akgec.ac.in*